# Apache Kafka Tutorial with Python

This notebook provides a comprehensive guide to working with Apache Kafka using Python.

## Table of Contents
1. Introduction to Kafka
2. Setting Up Kafka
3. Basic Producer
4. Basic Consumer
5. Advanced Producer with Callbacks
6. Advanced Consumer with Manual Commit
7. Stream Processing Example
8. Topic Administration

## ISR and Leader Election in Apache Kafka

Here’s a clean visual way to understand **ISR (In-Sync Replicas)** and **leader election** in Apache Kafka.

---

### 🧩 1. Normal State (Everything Healthy)

```
            Kafka Cluster
     ┌────────┬────────┬────────┐
     │Broker 1│Broker 2│Broker 3│
     └────────┴────────┴────────┘

Partition P0:

        Leader
          ↓
     [Broker 1]
        │
        │ replicate
        ↓
     [Broker 2]  ← Follower (ISR)
     [Broker 3]  ← Follower (ISR)
```

**Key points:**
- Leader = Broker 1
- ISR = {Broker 1, Broker 2, Broker 3}
- All replicas are fully synced ✔

---

### 🔁 2. How Replication Works

```
Producer → Broker 1 (Leader)

Broker 1 → sends data → Broker 2
         → sends data → Broker 3
```

- Followers constantly copy from leader
- If they stay up-to-date → remain in ISR

---

### ⚠️ 3. When a Follower Falls Behind

```
Partition P0:

     [Broker 1]  ← Leader
     [Broker 2]  ← In Sync
     [Broker 3]  ← ❌ Lagging (removed from ISR)

ISR = {Broker 1, Broker 2}
```

👉 Broker 3 is no longer trusted for leadership

---

### 💥 4. Leader Failure

```
❌ Broker 1 crashes

Before crash:
ISR = {Broker 1, Broker 2}
```

---

### 🔄 5. Leader Election

```
Controller selects new leader from ISR

     New Leader
         ↓
     [Broker 2]
        │
        ↓
     [Broker 3] (still catching up)
```

**Result:**
- Broker 2 becomes new leader
- System continues working ✅
- No data loss (because Broker 2 was in ISR)

---

### 🧠 Final Mental Model

```
ISR = "trusted, fully synced replicas"

Leader election rule:
👉 Only pick leader from ISR
```

---

### 🚀 One-Line Summary

- Leader handles traffic
- Followers replicate
- ISR = safe replicas
- If leader fails → ISR member becomes new leader


## Introduction to Kafka

**Apache Kafka** is a distributed event streaming platform used for building real-time data pipelines and streaming applications.

### Key Concepts:

- **Topic**: A category or feed name to which records are published (like a database table)
- **Producer**: Application that publishes records to Kafka topics
- **Consumer**: Application that subscribes to topics and processes records
- **Broker**: A Kafka server in the cluster
- **Partition**: Topics are divided into partitions for parallelism and scalability
- **Offset**: Unique identifier for each message in a partition
- **Consumer Group**: Group of consumers that share the workload

### Architecture:
```
Producer -> Topic (Partitioned) -> Consumer Group
           |
        Broker Cluster
```

## Setup

### Prerequisites:
1. Install kafka-python library:
```bash
pip install kafka-python pandas
```

2. Start Kafka using Docker:
```bash
docker-compose up -d
```

This will start:
- **Zookeeper** on port 2181 (manages Kafka cluster metadata)
- **Kafka Broker** on port 9092

## Basic Producer

A **Producer** sends data to Kafka topics. Let's create a producer that reads from our CSV dataset and publishes each row as a message.

In [ ]:
# Import required libraries
from kafka import KafkaProducer
import pandas as pd
import json
import time

# Load the dataset
df = pd.read_csv("data/ecommerce_dataset_updated.csv")
print(f"Loaded {len(df)} records from CSV")
df.head()

In [ ]:
# Create Kafka Producer
# bootstrap_servers: List of Kafka brokers to connect to
# value_serializer: Function to serialize message data (convert Python dict to JSON bytes)
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

print("Kafka Producer created successfully")

In [ ]:
# Define the topic name
topic_name = "sales"

# Send messages to Kafka
# producer.send() is asynchronous - it doesn't wait for acknowledgment
# producer.flush() ensures all messages are sent before continuing

print(f"Sending messages to topic: {topic_name}")

for idx, row in df.iterrows():
    message = row.to_dict()
    
    # Send message to Kafka
    future = producer.send(topic_name, message)
    
    # Print progress (first 5 messages)
    if idx < 5:
        print(f"Sent: {message}")
    
    # Small delay to simulate real-time streaming
    time.sleep(0.1)

# Ensure all messages are sent
producer.flush()

print(f"\nSuccessfully sent {len(df)} messages to Kafka")

In [ ]:
# Close the producer connection
producer.close()
print("Producer closed")

## Basic Consumer

A **Consumer** reads data from Kafka topics. Let's create a consumer that reads from the 'sales' topic.

In [ ]:
# Import KafkaConsumer
from kafka import KafkaConsumer
import json

# Create Kafka Consumer
consumer = KafkaConsumer(
    "sales",  # Topic to subscribe to
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",  # Start from the beginning (earliest) or end (latest)
    enable_auto_commit=True,  # Automatically commit offsets
    group_id="sales-group",  # Consumer group for load balancing   👉 A set of consumers working together to read a topic 
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))  # Deserialize JSON bytes to Python dict
)

print("Kafka Consumer created successfully")
print("Waiting for messages...")

In [ ]:
# Consume messages
# This will run indefinitely until interrupted (Ctrl+C)

message_count = 0
max_messages = 10  # Limit for demonstration

try:
    for message in consumer:
        # message.value contains the deserialized data
        print(f"Received: {message.value}")
        message_count += 1
        
        # Stop after receiving max_messages for demonstration
        if message_count >= max_messages:
            print(f"\nReceived {message_count} messages (stopping for demo)")
            break
            
except KeyboardInterrupt:
    print("\nConsumer interrupted by user")
finally:
    consumer.close()
    print("Consumer closed")

## Advanced Producer with Callbacks

Let's create a more robust producer with error handling and callbacks to track message delivery.

In [ ]:
from kafka import KafkaProducer
import json
import time

# Callback function to handle send completion
def on_send_success(record_metadata):
    """
    Called when a message is successfully sent to Kafka.
    record_metadata contains: topic, partition, offset, timestamp
    """
    print(f"Message sent to {record_metadata.topic} partition {record_metadata.partition} offset {record_metadata.offset}")

def on_send_error(excp):
    """
    Called when sending a message fails.
    """
    print(f"Error sending message: {excp}")

# Create producer with additional configurations
advanced_producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    acks="all",  # Wait for all replicas to acknowledge (strongest durability)
    retries=3,  # Retry up to 3 times on failure
    compression_type="gzip"  # Compress messages for efficiency
)

print("Advanced Producer created with callbacks and error handling")

In [ ]:
# Send messages with callbacks
sample_data = df.head(5).to_dict('records')

for data in sample_data:
    # Send with callbacks
    future = advanced_producer.send(
        "sales",
        value=data
    )
    
    # Add callbacks
    future.add_callback(on_send_success)
    future.add_errback(on_send_error)
    
    time.sleep(0.2)

# Flush to ensure all messages are sent
advanced_producer.flush()
print("\nAll messages sent with callbacks")

In [ ]:
# Close the producer
advanced_producer.close()
print("Advanced Producer closed")

## Advanced Consumer with Manual Commit

Manual offset commit gives you more control over when messages are marked as processed.

In [ ]:
from kafka import KafkaConsumer
import json

# Create consumer with manual commit
manual_commit_consumer = KafkaConsumer(
    "sales",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=False,  # Disable auto commit for manual control
    group_id="sales-manual-group",
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

print("Consumer with manual commit created")

In [ ]:
# Consume and manually commit offsets
message_count = 0
max_messages = 5

try:
    for message in manual_commit_consumer:
        data = message.value
        
        # Process the message
        print(f"Processing: {data.get('Product_ID', 'N/A')} - Rs.{data.get('Final_Price(Rs.)', 0)}")
        
        # Simulate processing
        time.sleep(0.1)
        
        # Manually commit offset after successful processing
        manual_commit_consumer.commit()
        print(f"Committed offset: {message.offset}")
        
        message_count += 1
        if message_count >= max_messages:
            break
            
except Exception as e:
    print(f"Error: {e}")
finally:
    manual_commit_consumer.close()
    print("\nManual commit consumer closed")

## Stream Processing Example

Let's create a real-time analytics consumer that calculates statistics from the stream.

In [ ]:
from kafka import KafkaConsumer
import json
from collections import defaultdict

# Create consumer for stream processing
analytics_consumer = KafkaConsumer(
    "sales",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="analytics-group",
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

print("Analytics Consumer created")

In [ ]:
# Real-time analytics processing
category_sales = defaultdict(float)
payment_methods = defaultdict(int)
total_revenue = 0
message_count = 0
max_messages = 20

print("Starting real-time analytics...\n")

try:
    for message in analytics_consumer:
        data = message.value
        
        # Extract fields
        category = data.get('Category', 'Unknown')
        price = float(data.get('Final_Price(Rs.)', 0))
        payment = data.get('Payment_Method', 'Unknown')
        
        # Update analytics
        category_sales[category] += price
        payment_methods[payment] += 1
        total_revenue += price
        message_count += 1
        
        # Print progress every 5 messages
        if message_count % 5 == 0:
            print(f"Processed {message_count} messages...")
        
        if message_count >= max_messages:
            break
            
except Exception as e:
    print(f"Error: {e}")
finally:
    analytics_consumer.close()
    
    # Print analytics summary
    print("\n" + "="*50)
    print("ANALYTICS SUMMARY")
    print("="*50)
    print(f"Total Messages Processed: {message_count}")
    print(f"Total Revenue: Rs.{total_revenue:.2f}")
    print(f"\nSales by Category:")
    for category, sales in category_sales.items():
        print(f"  {category}: Rs.{sales:.2f}")
    print(f"\nPayment Methods:")
    for method, count in payment_methods.items():
        print(f"  {method}: {count} transactions")
    print("="*50)

## Topic Administration

You can also create and manage topics programmatically using Kafka's AdminClient.

In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic

# Create admin client
admin_client = KafkaAdminClient(
    bootstrap_servers="localhost:9092",
    client_id="admin"
)

print("Kafka Admin Client created")

In [ ]:
# Create a new topic
topic_list = []

# Create topic: name, num_partitions, replication_factor
new_topic = NewTopic(
    name="sales-analytics",
    num_partitions=3,
    replication_factor=1
)

topic_list.append(new_topic)

# Create the topic
try:
    admin_client.create_topics(new_topics=topic_list, validate_only=False)
    print("Topic 'sales-analytics' created successfully")
except Exception as e:
    print(f"Topic may already exist or error: {e}")

In [ ]:
# List all topics
topics = admin_client.list_topics()
print(f"\nAvailable topics: {topics}")

In [ ]:
# Close admin client
admin_client.close()
print("\nAdmin Client closed")

## Summary

### Key Takeaways:

1. **Producer**: Sends messages to Kafka topics
   - Use `send()` for async sending
   - Use `flush()` to ensure delivery
   - Add callbacks for success/error handling

2. **Consumer**: Reads messages from Kafka topics
   - Use `auto_offset_reset` to control where to start
   - Use `group_id` for consumer groups
   - Manual commit gives more control

3. **Best Practices**:
   - Always close producers/consumers
   - Use appropriate acknowledgment settings
   - Handle errors gracefully
   - Use compression for large messages

### Next Steps:
- Explore Kafka Streams for complex stream processing
- Implement exactly-once semantics
- Set up Kafka Connect for data integration
- Monitor Kafka with tools like Kafka Manager